# ROGII Wellbore Geology - Transformer network implementation 

### Imports

In [1]:
# Cell 1: Environment & Imports
import os
os.environ["KERAS_BACKEND"] = "jax"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"] = "platform"

import jax
# Enable JAX float64 precision natively
jax.config.update("jax_enable_x64", True)

import keras
from keras import layers, callbacks, regularizers
import jax.numpy as jnp
import numpy as np
import polars as pl
import glob, pickle, warnings, random
warnings.filterwarnings("ignore")

# Show enough digits to round-trip float64 diagnostics
np.set_printoptions(precision=17, floatmode="unique")

# Set seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
keras.utils.set_random_seed(SEED)

print(f"Keras version : {keras.__version__}")
print(f"Keras backend : {keras.backend.backend()}")
print(f"Working dir   : {os.getcwd()}")


2026-07-31 22:33:28.315975: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1785526408.359133  161055 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1785526408.371714  161055 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Keras version : 3.12.0
Keras backend : jax
Working dir   : /home/samer/Documents/competitions/ROGII/notebooks


### Dataset Detection

In [2]:
# Cell 2: Auto-detect dataset path

def find_data_dir():
    """Searches common Kaggle mount points for the ROGII dataset."""
    candidates = [
        "/kaggle/input/competitions/rogii-wellbore-geology-prediction",
        "/kaggle/input/rogii-wellbore-geology-prediction",
        "/home/samer/Documents/competitions/ROGII/dataset",
    ]

    for scan_root in ["/kaggle/input", "/kaggle/input/competitions"]:
        if os.path.isdir(scan_root):
            for entry in os.listdir(scan_root):
                full = os.path.join(scan_root, entry)
                if os.path.isdir(full) and full not in candidates:
                    candidates.append(full)

    print("Searching for ROGII dataset...")
    for path in candidates:
        if not os.path.isdir(path):
            continue

        contents = os.listdir(path)
        has_train = "train" in contents and os.path.isdir(os.path.join(path, "train"))
        n_train = 0
        if has_train:
            n_train = len(glob.glob(os.path.join(path, "train", "*__horizontal_well.csv")))

        if n_train > 0:
            print(f"  V Using {path} (found {n_train} train wells)")
            return path

    raise FileNotFoundError("Could not find ROGII dataset.")

DATA_DIR = find_data_dir()


Searching for ROGII dataset...
  V Using /home/samer/Documents/competitions/ROGII/dataset (found 773 train wells)


### Data loading

In [ ]:
"""Data loading, preprocessing and preparation functions using Polars and JAX.

The Conv1D model is trained with a test-like TVT_input mask. In the competition test
wells, TVT_input is known before the submission interval and missing throughout the
interval that must be predicted. If validation keeps the true TVT_input, the model can
learn an identity shortcut and report unrealistically low validation RMSE.
"""

from typing import Dict, List, Tuple, Union
import glob
import os
import pickle
import jax
import jax.numpy as jnp
import numpy as np
import polars as pl

FEATURE_COLS: List[str] = ["MD", "X", "Y", "Z", "GR", "TVT_input"]
TARGET: str = "TVT"
TVT_INPUT_COL: str = "TVT_input"
MASK_START_RATIO_RANGE: Tuple[float, float] = (0.20, 0.40)


def mask_tvt_input_for_prediction_zone(
    df: pl.DataFrame,
    rng: np.random.Generator,
    ratio_range: Tuple[float, float] = MASK_START_RATIO_RANGE,
) -> Tuple[pl.DataFrame, int]:
    """Masks a suffix of TVT_input to mimic the hidden test prediction interval.

    Args:
        df (pl.DataFrame): Input DataFrame containing the well data.
        rng (np.random.Generator): Random number generator for reproducibility.
        ratio_range (Tuple[float, float]): Range of mask start ratio relative
          to well depth.

    Returns:
        Tuple[pl.DataFrame, int]: A tuple containing the masked DataFrame and
          the index where the masking started.
    """
    if TVT_INPUT_COL not in df.columns or len(df) < 2:
        return df, len(df)

    low, high = ratio_range
    mask_start = int(round(len(df) * rng.uniform(low, high)))
    mask_start = min(max(mask_start, 1), len(df) - 1)
    row_nr = pl.int_range(0, pl.len())
    df = df.with_columns(
        pl.when(row_nr >= mask_start)
        .then(None)
        .otherwise(pl.col(TVT_INPUT_COL))
        .alias(TVT_INPUT_COL)
    )
    return df, mask_start

def create_sequences(features: np.ndarray, target: np.ndarray, sequence_length: int) -> Tuple[np.ndarray, np.ndarray]:
    """Creates sequences from time series data.

    Args:
        features (np.ndarray): The input features.
        target (np.ndarray): The target values.
        sequence_length (int): The length of each sequence.

    Returns:
        A tuple of (sequence_features, sequence_targets).
    """
    X, y = [], []
    for i in range(len(features) - sequence_length + 1):
        X.append(features[i : (i + sequence_length)])
        y.append(target[i + sequence_length - 1])
    return np.array(X), np.array(y)

def preprocess(df: pl.DataFrame) -> pl.DataFrame:
    """Interpolates and fills null values in the feature columns.

    Args:
        df (pl.DataFrame): Input DataFrame with missing values.

    Returns:
        pl.DataFrame: Preprocessed DataFrame with all feature columns filled.
    """
    for col in FEATURE_COLS:
        if col in df.columns:
            df = df.with_columns(
                pl.col(col)
                .interpolate()
                .fill_null(strategy="forward")
                .fill_null(strategy="backward")
                .fill_null(0.0)
            )
    return df

def get_well_arrays(
    df: pl.DataFrame, rng: np.random.Generator
) -> Tuple[np.ndarray, np.ndarray, float]:
    """Preprocesses a well and returns feature/target arrays with TVT masking.

    Args:
        df (pl.DataFrame): The raw well DataFrame.
        rng (np.random.Generator): Random number generator for masking.

    Returns:
        A tuple of (features, targets, mask_start_ratio).
    """
    df_masked, mask_start = mask_tvt_input_for_prediction_zone(df, rng)
    df_proc = preprocess(df_masked)

    feats = df_proc.select(FEATURE_COLS).to_numpy().astype(np.float64)
    tgts = df_proc.select(TARGET).to_numpy().astype(np.float64).flatten()

    mask_ratio = mask_start / len(df) if len(df) > 0 else 0
    return feats, tgts, mask_ratio


def prepare_data(
    data_dir: str,
    val_ratio: float = 0.20,
    sequence_length: int = 64,
    stride: int = 8,
    seed: int = 42,
    max_wells: Union[int, None] = None,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, Dict]:
    """Loads wells and builds train/validation arrays with test-like masking.

    Reshapes the inputs to 3D tensors compatible with Conv1D layers and uses JAX
    on CPU memory for double-precision normalization to avoid GPU OOM.

    Args:
        data_dir (str): Path to the training dataset.
        val_ratio (float): Fraction of wells to use for validation.
        seed (int): Random seed for split and masking reproducibility.
        max_wells (Union[int, None]): Maximum number of wells to load (for debugging).

    Returns:
        Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, Dict]:
          A tuple containing:
            - X_train: Preprocessed standardized training features (3D: [samples, 1, features]).
            - y_train: Standardized training target values (1D: [samples]).
            - X_val: Preprocessed standardized validation features (3D: [samples, 1, features]).
            - y_val: Standardized validation target values (1D: [samples]).
            - y_val_raw: Raw validation targets in original scale (1D: [samples]).
            - scaler: Dictionary containing standardization scaling parameters.
    """
    dataset_root = os.path.dirname(data_dir)

    # 1. Get original training IDs
    pattern_train = os.path.join(data_dir, "train", "*__horizontal_well.csv")
    files_train = sorted([os.path.basename(f) for f in glob.glob(pattern_train)])

    # 2. Get augmented training files
    # pattern_aug = os.path.join(
    #     dataset_root, "dataset_augmented", "train", "*__horizontal_well_aug.csv"
    # )
    # files_aug = sorted([os.path.basename(f) for f in glob.glob(pattern_aug)])

    # Combined training file list (ensure full paths)
    files_train_all = files_train #+ files_aug

    if max_wells:
        files_train_all = files_train_all[:max_wells]

    rng = np.random.default_rng(seed)
    n_val = max(1, int(len(files_train_all) * val_ratio))
    val_set = set(rng.permutation(len(files_train_all))[:n_val])

    train_files = [fname for i, fname in enumerate(files_train_all) if i not in val_set]
    val_files = [fname for i, fname in enumerate(files_train_all) if i in val_set]

    print(
        f"Loading {len(files_train_all)} training wells total. "
        # f"(Original: {len(files_train)}, Augmented: {len(files_aug)})."
    )

    def get_split_data(files: List[str], split_name: str) -> Tuple[List[np.ndarray], List[np.ndarray]]:
        """Reads CSV files for a split, masks TVT_input, and returns lists of arrays.

        Args:
            files (List[str]): List of filenames to load.
            split_name (str): Label for the dataset split (e.g. "Train", "Val").

        Returns:
            Tuple[List[np.ndarray], List[np.ndarray]]: Lists of feature arrays and target arrays.
        """
        all_feats, all_tgts, all_mask_starts = [], [], []
        for i, filename in enumerate(files):
            try:
                # if filename in files_aug:
                #     path = os.path.join(
                #         dataset_root, "dataset_augmented", "train", filename
                #     )
                # else:
                path = os.path.join(data_dir, "train", filename)

                df = pl.read_csv(path, infer_schema_length=10000)
                df = df.filter(pl.col(TARGET).is_not_null())
                if len(df) == 0:
                    continue

                feats, tgts, mask_ratio = get_well_arrays(df, rng)
                all_feats.append(feats)
                all_tgts.append(tgts)
                all_mask_starts.append(mask_ratio)
            except Exception as e:
                print(f"  Skip {filename}: {e}")

        if not all_feats:
            raise ValueError(f"No usable wells found for {split_name} split.")

        print(
            f"{split_name} TVT_input mask start ratio: "
            f"mean={np.mean(all_mask_starts):.3f}, "
            f"min={np.min(all_mask_starts):.3f}, max={np.max(all_mask_starts):.3f}"
        )
        return all_feats, all_tgts

    # Load raw data for all wells
    X_train_wells, y_train_wells = get_split_data(train_files, "Train")
    X_val_wells, y_val_wells = get_split_data(val_files, "Val")

    # Concatenate all training data to compute global scaling factors
    X_train_flat_raw = np.concatenate(X_train_wells)
    y_train_flat_raw = np.concatenate(y_train_wells)

    # Use JAX on CPU for stable float64 normalization
    cpu_dev = jax.devices("cpu")[0]
    X_tr_flat_jax = jax.device_put(X_train_flat_raw, cpu_dev)
    y_tr_flat_jax = jax.device_put(y_train_flat_raw, cpu_dev)

    feat_mean = jnp.mean(X_tr_flat_jax, axis=0)
    feat_std = jnp.std(X_tr_flat_jax, axis=0)
    feat_std = jnp.where(feat_std == 0, 1.0, feat_std)

    target_mean = jnp.mean(y_tr_flat_jax)
    target_std = jnp.std(y_tr_flat_jax)
    target_std = jnp.where(target_std == 0, 1.0, target_std)

    # Normalize and create sequences for each well, then concatenate
    X_train_seqs, y_train_seqs = [], []
    for feats, tgts in zip(X_train_wells, y_train_wells):
        feats_n = (feats - feat_mean) / feat_std
        tgts_n = (tgts - target_mean) / target_std
        X_seq, y_seq = create_sequences(feats_n, tgts_n, sequence_length)
        X_train_seqs.append(X_seq)
        y_train_seqs.append(y_seq)

    X_val_seqs, y_val_seqs, y_val_raw_seqs = [], [], []
    for feats, tgts in zip(X_val_wells, y_val_wells):
        feats_n = (feats - feat_mean) / feat_std
        tgts_n = (tgts - target_mean) / target_std
        X_seq, y_seq = create_sequences(feats_n, tgts_n, sequence_length)
        _, y_raw_seq = create_sequences(feats, tgts, sequence_length)
        X_val_seqs.append(X_seq)
        y_val_seqs.append(y_seq)
        y_val_raw_seqs.append(y_raw_seq)

    # Final concatenation into large numpy arrays
    X_train = np.concatenate(X_train_seqs).astype(np.float32)
    y_train = np.concatenate(y_train_seqs).astype(np.float32)
    X_val = np.concatenate(X_val_seqs).astype(np.float32)
    y_val = np.concatenate(y_val_seqs).astype(np.float32)
    y_val_raw = np.concatenate(y_val_raw_seqs).astype(np.float64)

    # Shuffle the training data
    p = rng.permutation(len(X_train))
    X_train, y_train = X_train[p], y_train[p]

    # The validation set for evaluation should not be shuffled to preserve well structure if needed later
    scaler = {
        "feature_cols": FEATURE_COLS,
        "feat_mean": np.array(feat_mean),
        "feat_std": np.array(feat_std),
        "target_mean": float(target_mean),
        "target_std": float(target_std),
        "normalized": True,
        "tvt_input_masked_for_training": True,
        "mask_start_ratio_range": MASK_START_RATIO_RANGE,
    }

    return (
        X_train,
        y_train,
        X_val,
        y_val,
        y_val_raw,
        scaler,
    )


print("Preparing dataset...")
X_train, y_train, X_val, y_val, y_val_raw, scaler = prepare_data(
    DATA_DIR,
    sequence_length=32,  # Using a sequence length of 64
    stride=4             # A stride of 8 makes training faster
)
print(f"Train size: {X_train.shape}, Val size: {X_val.shape}")

OUT_DIR = "../outputs"
os.makedirs(OUT_DIR, exist_ok=True)
scaler_path = os.path.join(OUT_DIR, "conv_tr_scaler_params_v3.pkl")
with open(scaler_path, "wb") as f:
    pickle.dump(scaler, f)
print(f"Scaler parameters saved to {scaler_path}")


Preparing dataset...
Loading 773 training wells total. 
Train TVT_input mask start ratio: mean=0.301, min=0.200, max=0.400
Val TVT_input mask start ratio: mean=0.295, min=0.200, max=0.399


### Model building

In [ ]:
# Cell 7: Transformer Model Definition

def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0):
    """Creates a single transformer block."""
    # Attention and Normalization
    x = layers.LayerNormalization(epsilon=1e-6)(inputs)
    x = layers.MultiHeadAttention(
        key_dim=head_size, num_heads=num_heads, dropout=dropout
    )(x, x)
    x = layers.Dropout(dropout)(x)
    res = x + inputs

    # Feed Forward Part
    x = layers.LayerNormalization(epsilon=1e-6)(res)
    x = layers.Conv1D(filters=ff_dim, kernel_size=1, activation="relu")(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Conv1D(filters=inputs.shape[-1], kernel_size=1)(x)
    return x + res

def build_transformer_model(
    input_shape,
    head_size,
    num_heads,
    ff_dim,
    num_transformer_blocks,
    mlp_units,
    dropout=0,
    mlp_dropout=0,
):
    """Builds the full Transformer-based model."""
    inputs = keras.Input(shape=input_shape)
    x = inputs
    for _ in range(num_transformer_blocks):
        x = transformer_encoder(x, head_size, num_heads, ff_dim, dropout)

    x = layers.GlobalAveragePooling1D(data_format="channels_last")(x)
    for dim in mlp_units:
        x = layers.Dense(dim, activation="relu")(x)
        x = layers.Dropout(mlp_dropout)(x)
    outputs = layers.Dense(1)(x)

    return keras.Model(inputs, outputs)

# Get input shape from the preprocessed data
input_shape = X_train.shape[1:]

# Build the model with specified hyperparameters
transformer_model = build_transformer_model(
    input_shape,
    head_size=128,
    num_heads=4,
    ff_dim=4,
    num_transformer_blocks=4,
    mlp_units=[128],
    dropout=0.1,
    mlp_dropout=0.1,
)

# Compile the model with Adam optimizer and Mean Squared Error loss
transformer_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss="mean_squared_error",
    metrics=["mean_absolute_error", "root_mean_squared_error"],
)

transformer_model.summary()

In [ ]:
# Cell 8: Model Training

# Define callbacks for training
checkpoint_path = os.path.join(OUT_DIR, "transformer_v1.keras")
model_checkpoint = callbacks.ModelCheckpoint(
    checkpoint_path,
    monitor="val_root_mean_squared_error",
    save_best_only=True,
    mode="min",
    verbose=1,
)
early_stopping = callbacks.EarlyStopping(
    monitor="val_root_mean_squared_error",
    patience=10,
    mode="min",
    verbose=1,
    restore_best_weights=True,
)

# Train the model
history = transformer_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=256,
    callbacks=[model_checkpoint, early_stopping],
    verbose=1,
)

In [ ]:
# Cell 9: Model Evaluation

# Load the best performing model from the checkpoint
print(f"Loading best model from: {checkpoint_path}")
best_model = keras.models.load_model(checkpoint_path)

# Make predictions on the validation set
y_pred_normalized = best_model.predict(X_val)

# Inverse transform the predictions and true values to their original scale
y_pred_original = (y_pred_normalized.flatten() * scaler["target_std"]) + scaler["target_mean"]
y_val_original = y_val_raw # Use the raw validation targets saved during preprocessing

# Calculate RMSE on the original scale
final_rmse = np.sqrt(np.mean((y_pred_original - y_val_original)**2))
print(f"\nFinal Validation RMSE (Original TVT Scale): {final_rmse:.4f}")

# Plot training history
def plot_history(history):
    plt.style.use("dark_background")
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

    # Plot loss
    ax1.plot(history.history["loss"], label="Train Loss")
    ax1.plot(history.history["val_loss"], label="Val Loss")
    ax1.set_title("Model Loss (Normalized Scale)")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss (MSE)")
    ax1.legend()

    # Plot RMSE
    ax2.plot(history.history["root_mean_squared_error"], label="Train RMSE")
    ax2.plot(history.history["val_root_mean_squared_error"], label="Val RMSE")
    ax2.set_title("Model RMSE (Normalized Scale)")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("RMSE")
    ax2.legend()

    plt.tight_layout()
    plt.show()

plot_history(history)